# Sampling implementation tests

In [ ]:
"""
Sampling script for generating exact samples from planar Ising models using
the planar_ising package.

Place this file inside the planar_ising-master/tests/ directory so that the
package imports and relative paths used in this script work correctly.

The script:
- constructs an L1 x L2 square lattice with open boundary conditions,
- generates a task-specific spin-glass realization with Jij in {-1, +1},
- scales the couplings by the specified inverse temperature beta,
- samples spin configurations using InferenceAndSampling,
- restarts the sampler with a fresh random stream if sampling stalls,
- saves the generated samples and empirical moments to the results directory.

For SLURM array jobs, SLURM_ARRAY_TASK_ID is used to generate a different,
reproducible Jij realization for each task.
"""

In [1]:
import os
import signal
import time
import numpy as np
import h5py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from planar_ising import PlanarIsingModelGenerator, PlanarGraphConstructor, InferenceAndSampling, PlanarIsingModel

In [ ]:
Path.cwd()

PosixPath('/Users/sbiswal/Documents/Code/planar_ising-master/tests')

In [2]:
def square_lattice_ordered_adjacencies(L1, L2):
    """
    Create ordered_adjacencies for an L1 x L2 square lattice
    for PlanarGraphConstructor.construct_from_ordered_adjacencies().

    Vertices are indexed row-major:
        v = r * L2 + c
    with r in {0, ..., L1-1}, c in {0, ..., L2-1}.

    Neighbor order is clockwise:
        right, down, left, up
    with missing boundary neighbors skipped.
    """
    def idx(r, c):
        return r * L2 + c

    ordered_adjacencies = []

    for r in range(L1):
        for c in range(L2):
            nbrs = []

            # right
            if c + 1 < L2:
                nbrs.append(idx(r, c + 1))

            # down
            if r + 1 < L1:
                nbrs.append(idx(r + 1, c))

            # left
            if c - 1 >= 0:
                nbrs.append(idx(r, c - 1))

            # up
            if r - 1 >= 0:
                nbrs.append(idx(r - 1, c))

            ordered_adjacencies.append(nbrs)

    return ordered_adjacencies

In [3]:
def all_configurations(n):
    """
    Return Sigma with shape (2^n, n), entries in {-1, +1}.
    Row idx corresponds to configuration index idx.
    Least Significant Bit corresponds to spin/node 0.
    """
    n_states = 1 << n
    Sigma = np.empty((n_states, n), dtype=np.int8)

    for idx in range(n_states):
        for j in range(n):
            Sigma[idx, j] = 1 if ((idx >> j) & 1) else -1

    return Sigma


def log_prob_config(x, beta, edges, edge_weights):
    """
    log unnormalized probability for one configuration.
    Assumes edges are 0-based.
    """
    s = 0.0

    for (i, j), w in zip(edges, edge_weights):
        s += w * float(x[i]) * float(x[j])

    return beta * s


def true_moments(Sigma, beta, edges, edge_weights):
    """
    Computes:
    - p_unnorm: unnormalized probabilities
    - Z       : partition function
    - mean    : true mean vector
    - cov     : true covariance matrix
    """

    Sigma = np.asarray(Sigma, dtype=np.int8)

    n_configs, n = Sigma.shape

    log_p = np.empty(n_configs, dtype=float)

    for l in range(n_configs):
        log_p[l] = log_prob_config(Sigma[l, :], beta, edges, edge_weights)

    # Log-sum-exp trick
    log_p_max = np.max(log_p)
    log_Z = log_p_max + np.log(np.sum(np.exp(log_p - log_p_max)))
    Z = np.exp(log_Z)

    p_unnorm = np.exp(log_p)
    p_norm = np.exp(log_p - log_Z)

    # Mean
    mean = np.sum(Sigma * p_norm[:, None], axis=0)

    # Second moment E[xx^T]
    M2 = (Sigma.T * p_norm) @ Sigma

    # Covariance
    cov = M2 - np.outer(mean, mean)

    return p_unnorm, Z, mean, cov


def empirical_moments(samples):
    freqs = samples[:, 0]
    spins = samples[:, 1:]

    w = freqs / np.sum(freqs)

    mean = spins.T @ w
    cov = (spins.T * w) @ spins - np.outer(mean, mean)

    return mean, cov


def generate_lattice_graph(rows, cols):
    """
    Generates edges as tuples for a lattice graph.
    """
    edges = []

    for r in range(rows):
        for c in range(cols):
            node = r * cols + c

            # Connect to right neighbor
            if c < cols - 1:
                right = node + 1
                edges.append((node, right))

            # Connect to bottom neighbor
            if r < rows - 1:
                below = node + cols
                edges.append((node, below))

    return edges

In [4]:
def compress_samples(samples):
    unique_configs, counts = np.unique(samples, axis=0, return_counts=True)
    compressed = np.hstack((counts[:, None], unique_configs))

    return compressed.astype(np.int64)

In [ ]:
# L by L square lattice
L1, L2 = 5,5
N = L1 * L2
n_edges = 2*L1* L2 - L1 - L2

# --- construct graph topology and the Ising model --- 
ordered_adjacencies = square_lattice_ordered_adjacencies(L1, L2)
graph = PlanarGraphConstructor.construct_from_ordered_adjacencies(ordered_adjacencies)

# One interaction per edge
J = np.ones(n_edges, dtype=float) # array([0.5, -0.2, 0.8, 0.1], dtype=float)

# rng = np.random.default_rng(seed=201)
# J = rng.choice([-1.0, 1.0], size=n_edges)

beta = 0.8

model = PlanarIsingModel(graph, beta*J)

In [ ]:
class SamplingTimeout(Exception):
    pass

def timeout_handler(signum, frame):
    raise SamplingTimeout

signal.signal(signal.SIGALRM, timeout_handler)

M = 20_000
timeout_seconds = 5

samples8 = np.empty((M, N), dtype=np.int8)

i = 0

while i < M:

    # Recreate sampler after a timeout
    inference = InferenceAndSampling(model)
    inference.prepare(sampling=True)

    # Start a new random stream
    np.random.seed(None)

    last_success = time.monotonic()

    while i < M:

        remaining = timeout_seconds - (time.monotonic() - last_success)

        if remaining <= 0:
            print(f"\nNo successful sample for {timeout_seconds} seconds.")
            break

        try:
            signal.setitimer(signal.ITIMER_REAL, remaining)

            samples8[i, :] = inference.sample_spin_configurations(1)[0]

            i += 1
            last_success = time.monotonic()

            print(f"\rProgress: {i}/{M}", end="", flush=True)

        except (AssertionError, ValueError):
            continue

        except SamplingTimeout:
            print(f"\nNo successful sample for {timeout_seconds} seconds.")
            break

        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)

    if i < M:
        print(f"Restarting sampler from {i}/{M}...")

print(f"\nCompleted {i}/{M} samples.")

Progress: 20000/20000
Completed 20000/20000 samples.


In [ ]:
# samples2.shape
# concatenate
data_samples = np.vstack((samples, samples1, samples2, samples3, samples4, samples5, samples6, samples7, samples8, samples9))
data_samples.shape

(200000, 25)

In [ ]:
# save
import csv
np.savetxt("5x5_M=200K_SpinGlass_beta=1.csv", data_samples, delimiter=",", fmt="%g")

In [ ]:
# import h5py

# L1, L2 = 5,5
# beta=0.6

# edges = generate_lattice_graph(L1, L2)
with h5py.File("../../Autoregressive Models/Data/5x5/Data_5x5_Jij=pm1_beta=06_t10.h5", "r") as f:
    J = f["edge_weights"][:]   # loads the dataset as a NumPy array
print(J)

# with h5py.File("../../Autoregressive Models/Data/5x5/configuration_list_5x5.h5", "r") as f:
#     Sigma = f["Sigma"][()]

p_unnorm, norm_const, mean_true, cov_true = true_moments(Sigma, beta, edges, J)

with h5py.File("Data_5x5_SpinGlass_beta=06_t10.h5", "w") as f:
    f.create_dataset("edge_weights", data=J)
    f.create_dataset("p_unnorm", data=p_unnorm)
    f.create_dataset("norm_const", data=norm_const)
    f.create_dataset("mean_true", data=mean_true)
    f.create_dataset("cov_true", data=cov_true)


[ 1. -1. -1.  1. -1.  1.  1.  1. -1.  1. -1.  1.  1.  1. -1.  1.  1.  1.
 -1. -1.  1.  1. -1.  1. -1. -1.  1.  1.  1.  1.  1. -1. -1.  1.  1. -1.
  1.  1.  1. -1.]


In [19]:
# Read and Edit
import h5py

with h5py.File("../../Autoregressive Models/Data/5x5/Data_5x5_SpinGlass_beta=08_t10.h5", "r") as f:
    J = f["edge_weights"][:]
    p_old = f["p_unnorm"][:]

print(J)

# --- Generate True Moments ---
L1, L2 = 5,5
N = L1 * L2
n_edges = 2*L1* L2 - L1 - L2
edges = generate_lattice_graph(L1, L2)
beta = 0.8

with h5py.File("../../Autoregressive Models/Data/5x5/configuration_list_5x5.h5", "r") as f:
    Sigma = f["Sigma"][()]

p_unnorm, norm_const, mean_true, cov_true = true_moments(Sigma, beta, edges, J)

with h5py.File("Data_5x5_SpinGlass_beta=08_t10.h5", "w") as f:
    f.create_dataset("edge_weights", data=J)
    f.create_dataset("p_unnorm", data=p_unnorm)
    f.create_dataset("norm_const", data=norm_const)
    f.create_dataset("mean_true", data=mean_true)
    f.create_dataset("cov_true", data=cov_true)

[ 1.  1.  1.  1. -1.  1.  1.  1. -1. -1. -1.  1.  1. -1.  1.  1.  1. -1.
  1. -1. -1.  1.  1. -1.  1. -1.  1.  1. -1. -1.  1.  1. -1. -1. -1.  1.
 -1.  1. -1. -1.]


In [ ]:
# # --- Generate Empirical Moments ---
# import h5py
# import csv 

# edges = generate_lattice_graph(L1, L2)

# data_samples = np.loadtxt("../../Autoregressive Models/Data/10x10/Samples_10x10_M=200K_Jij=1_beta=06.csv", delimiter=",", dtype=float)

# mean_true, cov_true = empirical_moments(compress_samples(data_samples))

# with h5py.File("Data_10x10_Jij=1_beta=06.h5", "w") as f:
#     # f.create_dataset("edge_weights", data=J)
#     f.create_dataset("mean_true", data=mean_true)
#     f.create_dataset("cov_true", data=cov_true)
